In [1]:
#download and install langchain dependencies
!pip install -qU langchain-community faiss-cpu langchain-openai

In [2]:
# importing all libraries needed
import faiss #for faiss vector database
from langchain_community.docstore.in_memory import InMemoryDocstore #for allocating vector database storage (in RAM)
from langchain_community.vectorstores import FAISS #for langchain - faiss vector database connector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI #for embedding and LLM service
from sentence_transformers import CrossEncoder #import crossencoder for reranker
from pydantic import BaseModel, Field #pydantic BaseModel and Field to create a data validation and data type definition
from typing import List #for list type

In [3]:
import getpass #for input password interface
import os #for read and manage python directory path and environment
from google.colab import userdata

#input required credentials
if not os.environ.get("OPENAI_API_KEY"):
  try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
  except:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [4]:
#define embedding model and llm from openai provider
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini")

In [5]:
#define path for load faiss index
path_faiss = "/content/drive/MyDrive/JCAIEAH-003/Notes and Hands On/Modul 3/Day 10/faiss_index"

#create vector store with saved index
vector_store = FAISS.load_local(
    path_faiss, embeddings, allow_dangerous_deserialization=True
)

In [10]:
#retrieve relevant documents from vector database
query="Men's jacket"
documents = vector_store.similarity_search_with_relevance_scores(query=query, k=10)
documents

[(Document(id='e0f0e5d360b14e137e5b438cd99f5849', metadata={'uniq_id': 'e0f0e5d360b14e137e5b438cd99f5849', 'product_name': "Lee Mark Men's Solid Formal Shirt"}, page_content="Lee Mark Men's Solid Formal Shirt\n                         Price: Rs. 599\n\t\t\t\t\n\t\t\tLee Mark Brand Casual Shirts Are A Fashion Essential This Season And This Uber Cool Piece Is Just The Right Pick. This Shirt Features Long Sleeves, Ponit Collar With A Full Buttoned Chest Placket For Added Ease And A Curved Hem That Completes The Look. Pair This Piece With Chinos And Loafers For A Stylish Laid-Back Look.\nLee Mark Brand Casual Shirts Are A Fashion Essential This Season And This Uber Cool Piece Is Just The Right Pick. This Shirt Features Long Sleeves, Ponit Collar With A Full Buttoned Chest Placket For Added Ease And A Curved Hem That Completes The Look. Pair This Piece With Chinos And Loafers For A Stylish Laid-Back Look."),
  np.float32(0.25828743)),
 (Document(id='fd41178d3e845364fce52914580fd852', metada

#Pre-Trained Reranker Model

In [7]:
# Load the model, here we use the model
model = CrossEncoder("mixedbread-ai/mxbai-rerank-large-v1")

config.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

In [8]:
docs = [d.page_content for d in documents] #get page_content for each document
results = model.rank(query, docs, return_documents=True, top_k=3) #use reranker and take top-3
results

[{'corpus_id': 7,
  'score': np.float32(0.4038086),
  'text': "Key Features of Wake Up Competition Full Sleeve Striped Men's Sweatshirt Suitable For: Western Wear Hooded Pockets: Kangaroo Pockets at Front,Wake Up Competition Full Sleeve Striped Men's Sweatshirt Price: Rs. 748 To perfect an easy going look, you must get this navy coloured hoodie from Wake Up Competition. fleece fabric, this trendy sweatshirt will be one of your smartest and worthiest picks till date. Team it over a pair of blue jeans complemented by sneakers to attain a chilled out look when you go to hang out with friends.,Specifications of Wake Up Competition Full Sleeve Striped Men's Sweatshirt Sweatshirt Details Sleeve Full Sleeve Hooded Yes Reversible No Fabric Fleece Pockets Kangaroo Pockets at Front General Details Pattern Striped Occasion Casual Ideal For Men's In the Box 1 Sweatshirt Additional Details Style Code 16-725-NAVY Fabric Care Do Not Bleach, Wash with Similar Colors, Gentle Machine Wash Cold at 30?C, 

#LLM As Reranker

In [11]:
#create prompt to make llm acts and responses as a reranker
prompt = f"""
Query: {query}

Documents:
{docs}

Task: Rank the documents from most to least relevant to the query.
"""
response = llm.invoke(prompt) #generate response from llm
print(response.content) #show the result

Based on the query "Men's jacket," the documents can be ranked by relevance as follows:

1. **Specifications of Mizuno Lightweight Hoody Solid Men's Wind Cheater**: This document is the most relevant as it describes a men's windcheater, which falls under the category of jackets. It details the features and specifications relevant to outerwear.
  
2. **Key Features of Roadster Men's Zipper Solid Cardigan**: This document describes a men's cardigan, which is also an outerwear piece. It's relevant but slightly less so than the first document since cardigans are typically considered less formal than jackets but still serve the purpose of providing warmth.

3. **Wake Up Competition Full Sleeve Striped Men's Sweatshirt**: This document discusses a men's sweatshirt. While it's not a jacket, it serves a similar purpose in providing warmth.

4. **Lee Mark Men's Solid Formal Shirt** (Two identical entries): These documents are not relevant to the jacket query as they describe men's shirts, which

In [12]:
#make llm as reranker responses with structured output
class DocumentAttribute(BaseModel): #detail information for responses
  doc_number: str = Field(..., description="Original document number")
  score: float = Field(..., description="relevant score, interval 0 to 1")

class ResponseOutput(BaseModel): #output template
  reranker_result: List[DocumentAttribute] = Field(..., description="reranker result") #template from DocumentAttribute class for detail of each document


llm_with_structured__output = llm.with_structured_output(ResponseOutput) #method to give information and instruction to llm to follow the output template
response = llm_with_structured__output.invoke(prompt) #generate response
response.model_dump() #transform the output from pydantic object into json/dictionary


{'reranker_result': [{'doc_number': '3', 'score': 0.9},
  {'doc_number': '4', 'score': 0.8},
  {'doc_number': '1', 'score': 0.2},
  {'doc_number': '2', 'score': 0.2},
  {'doc_number': '5', 'score': 0.1},
  {'doc_number': '6', 'score': 0.1},
  {'doc_number': '7', 'score': 0.1},
  {'doc_number': '8', 'score': 0.1},
  {'doc_number': '9', 'score': 0.1}]}